In [ ]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import KFold
from sklearn.utils import resample

# Load Iris dataset
iris = load_iris()
X, y = iris.data, iris.target
model = DecisionTreeClassifier(random_state=42)

# -------- Bootstrapping --------
boot_accuracy, boot_f1 = [], []

for i in range(10):
    X_boot, y_boot = resample(
        X, y, replace=True, random_state=i
    )

    # Find out-of-bag samples
    indices = resample(
        range(len(X)), replace=True, random_state=i
    )
    oob = list(set(range(len(X))) - set(indices))

    if not oob:
        continue

    # Train and test
    model.fit(X_boot, y_boot)
    pred = model.predict(X[oob])

    boot_accuracy.append(accuracy_score(y[oob], pred))
    boot_f1.append(f1_score(y[oob], pred, average='weighted'))

print("Bootstrapping Results")
print("Average Accuracy:", round(np.mean(boot_accuracy), 4))
print("Average F1 Score:", round(np.mean(boot_f1), 4))

# -------- 5-Fold Cross-Validation --------
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_accuracy, cv_f1 = [], []

for train, test in kf.split(X):
    model.fit(X[train], y[train])
    pred = model.predict(X[test])

    cv_accuracy.append(accuracy_score(y[test], pred))
    cv_f1.append(f1_score(y[test], pred, average='weighted'))

print("\nK-Fold Cross Validation Results")
print("Average Accuracy:", round(np.mean(cv_accuracy), 4))
print("Average F1 Score:", round(np.mean(cv_f1), 4))